---
format: 
    html:
      toc: true
title: "Raw data alignment"
author: "Stig U Andersen, Mikkel H Schierup, Samuele Soraggi"
date-modified: last-modified
title-block-banner: true
---

:::{.callout-note title="Tutorial description"}

This tutorial will cover the steps for performing the alignment of raw RNA- and HiFi-sequencing data. You will need to use [the software IGV](https://software.broadinstitute.org/software/igv/download) on your computer to visualize some of the output files, which can be easily downloaded once they are produced. At the end of this tutorial you will be able to:

- perform and discuss quality control on raw data in `fastq` format using `FastQC` and `MultiQC`
- align HiFi and RNA sequencing data with dedicated tools such as `MiniMap2` and `STAR`
- analyze the quality the alignment with `qualimap`

The output of this notebook will be used for the Variant calling analysis and the bulk RNA-sequencing analysis.
:::



# Biological background

White clover (*Trifolium repens*) is a common plant found in fields across the world. It has an unusual origin story: it's the result of two different plant species "merging" their genetic material. At some point during evolution (during the last ice age), these two diploid (2n) parent species—*T. occidentale* and *T. pallescens*—naturally hybridized and created white clover, which is in result allotetraploid (4n) (see figure below). 

<figure>
<img src="images/white_clover.png" width="700" alt="Kernel Choice" class="center">
</figure>

- Normal plants have two copies of each gene: one from maternal origin, one from paternal origin (these plants are *diploid*, 2n)
- White clover, however has **four copies** of each gene. That is, two copies inherited from *T. occidentale* and two from *T. pallescens* (called an *allotetraploid*, 4n)
- These four copies are in the same nucleus, you can think of it as two complete genomes side-by-side.

Normally, white clover  is an obligate outcrosser species—that means that plants swap pollen with other plants instead of fertilizing themselves. However, a special self-compatible (it *can* self-fertilize) line was used for sequencing its genome [(Griffiths et al, 2019)](https://academic.oup.com/plcell/article/31/7/1466/5985684). This line is designated as `S10` in our data (this is the 10th self-fertilized generation). In addition, we also have data from a wild clover variety (ecotype) called Tienshan (`Ti`), from mountains in China. This variety is adapted to alpine conditions, making it genetically different from the S10 line.

We have sequences (DNA "reads") from both clover varieties, and we need to align them to the white clover reference genome. However, our reference genome is tricky: it contains sequences from **both parent species** (we call them `contig 1` and `contig 2`). Therefore, when we align short DNA reads to the reference, some reads might match equally well to both `contig 1` and `contig 2` (they're similar but different species). 

We'll use quality control tools to: (1) Align reads to the **complete reference** (both contigs together); (2) Align reads to **each subgenome separately** (`contig 1` alone, `contig 2` alone); and (3) Compare the results to see how this affects our data quality and interpretation

# Quality control and mapping

## Quality Control

We run `FastQC` on the PacBio Hifi reads and on two of the Illumina RNA-seq libraries. `FastQC` does quality control of the raw sequence data, providing an overview of the data which can help identify if there are any problems that should be addressed before further analysis. You can find the report for each file into the folder `results/fastqc_output/`. The output is in HTML format and can be opened in any browser or in `jupyterlab`. It is however not easy to compare the various libraries by opening separate reports. To aggregate all the results, we apply the `MultiQC` software to the reports' folder. The output of MultiQC is in the directory `results/multiqc_output/fastqc_data`.

In [ ]:
%%bash
#run fastqc
mkdir -p results/fastqc_output
fastqc -q -o results/fastqc_output ../Data/Clover_Data/*.fastq  > /dev/null 2>&1
echo "Done \n Files: $(ls ../Data/Clover_Data/*.fastq)"

**Note:** `fastqc` prints a lot of output conisting of a simple confirmation of execution without error, even when using the option `-q`, which means `quiet`. Therefore we added `> /dev/null 2>&1` to the command to mute the printing of that output.

In [ ]:
%%bash
#run multiqc
mkdir -p results/multiqc_output/fastqc_data
multiqc --outdir results/multiqc_output/fastqc_data results/fastqc_output

:::{.callout-tip title="Questions"}

Visualize the Webpage generated by MultiQC.

Hint: You can find a Help button that offers additional information about the plots for each panel. Focus on the following panels: “Per base sequence quality”, “Per sequence quality scores”.... (“Per base sequence content” always gives a FAIL for RNA-seq data).

Look at the sequence quality scores: 

- is there a marked difference between HiFi and illumina data?
- What do you notice with respect to the sequence quality scores? And are there any other quality issues worth noting?
- Is there anything off in the GC content? Why?
- Think about why “Per base sequence content” always fails. Can you imagine how the nucleotide content is biased because of priming bias during cDNA synthesis and expressed transcripts in the sample?

:::

## Hifi data mapping — long DNA reads

We map the PacBio Hifi reads (`Hifi_reads_white_clover.fastq`) to the white clover reference sequence (Contig1&2) using `minimap2`.

To demonstrate how mapping algorithm choice can heavily affect results, we deliberately run the alignment twice with two different preset options (`-x` flag):

* `map-hifi`: optimized for long reads (PacBio HiFi data)
* `sr`: optimized for short reads (Illumina data) 


The `map-hifi` setting expects long reads and employ algorithms that **handle large insertions, deletions, and other structural variations more effectively**. The scoring and alignment thresholds are adjusted to account for the longer sequence context.
The `sr` setting xpect shorter reads and optimize for quick, efficient alignment of these short sequences. The **focus is on minimizing mismatches and handling the dense packing of short reads**.

The `map-hifi` setting is designed for long reads and is more lenient with gaps and mismatches typical of long-read sequencing. The `sr` setting assumes short reads and is stricter about matches. By comparing both, you can see how choosing the wrong algorithm/option can affect your results—an important lesson in bioinformatics. **know your options**.

Next, we create reports of the mapping results by running `QualiMap` on the two obtained SAM files.

We first need to index the reference fasta files using `samtools faidx`. This produces files in `.fai` format containing informations about length of the reference sequence, offset for the quality scores, name of the reference sequence. [Click here](http://www.htslib.org/doc/faidx.html) for a detailed overview of `fai` format. 

In [ ]:
%%bash
#copy the reference data in the folder reference_data, so that you can write the indexing files
mkdir -p reference_data
cp ../Data/Clover_Data/DNA_Contig1_2.fasta ../Data/Clover_Data/DNA_Contig1.fasta ../Data/Clover_Data/DNA_Contig2.fasta reference_data

In [ ]:
%%bash
samtools faidx reference_data/DNA_Contig1_2.fasta
samtools faidx reference_data/DNA_Contig1.fasta
samtools faidx reference_data/DNA_Contig2.fasta

we create an output folder for the HIFI alignment, and run `minimap2` with the settings explained before.

In [ ]:
%%bash 
mkdir -p results/HIFI_alignment/
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sam \
                            reference_data/DNA_Contig1_2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq 

minimap2 -a -x sr -o results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sam \
                            reference_data/DNA_Contig1_2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

`samtools sort` is used to sort the alignment with left-to-right coordinates. The output is in `.bam` format, with `.sam` files in input (Note that you could have gotten `.bam` files from `minimap2` with a specific option).

In [ ]:
%%bash
samtools sort results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sam \
                -o results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam

samtools sort results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sam \
                -o results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam

`samtools index` creates the index for the `bam` file, stored in `.bai` format. The index file lets programs access any position into the aligned data without reading the whole file, which would take too much time.

In [ ]:
%%bash
samtools index results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam
samtools index results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam

:::{.callout-tip title="Questions - IGV"}

Now you can inspect the alignment files in IGV.

- First, you need the references fasta sequence in `../Data/Clover_Data/DNA_Contig1_2.fasta`, `../Data/Clover_Data/DNA_Contig1.fasta`, `../Data/Clover_Data/DNA_Contig2.fasta` imported into IGV. Remember: this is done with the menu `Genomes --> Load Genome from file` and by selecting the relevant fasta file. Then, choose the reference you need from the drop-down menu (see figure below).
You will not yet see much, but you can choose one of the two subgenomes (contig 1 or 2) and double click on a chromosome position to inspect the reference sequence. The next step will visualize the mapped files on IGV.

&nbsp;

- Choose the reference `DNA_Contig1_2.fasta` and load the two `.bam` files obtained with the two different alignment settings (`results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam`, `results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam`) to show them against the reference. This is done by choosing the `.bam` files from the `File --> Load from file` menu.

    - What do you notice about them? What is characteristic about the alignment `sr`?
    - Can you link these characteristics to the quality report seen before?
    - How did the alignment forced itself to align the reads for `sr` settings? Do you see a lot of mismatches and indels in the `sr` alignment?

:::

Run quality control on both files

In [ ]:
%%bash
unset DISPLAY

qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam \
                 -outdir results/qualimap_output/PacBio_clover_alignment_1_2_maphifi

qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam \
                 -outdir results/qualimap_output/PacBio_clover_alignment_1_2_sr

For easier comparison, we can again collapse the two reports into a single one using `MultiQC`, in the same way we did for putting together the other reports from `fastQC`.

In [ ]:
%%bash

#run multiqc
multiqc --outdir results/qualimap_output results/qualimap_output

:::{.callout-tip title="Questions"}

Now you can visualize the report generated, which is in `results/qualimap_output/multiqc_report.html`

- How do the average coverage of each sample correspond to what you have seen in the alignments using IGV?
- How comes that the GC content remains the same across the different alignments?

:::

Next, we map the white clover PacBio Hifi reads to contig1 and contig2 separately, using the setting you selected at the previous step (let's say `map-hifi` was chosen, but you are free to change this setting in the commands). As the two contigs represent the two white clover subgenomes, this mapping will allow you to see the two subgenome haplotypes and call subgenome SNPs.


In [ ]:
%%bash 
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1.sam \
                            reference_data/DNA_Contig1.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

In [ ]:
%%bash 
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_2.sam \
                            reference_data/DNA_Contig2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

Sort the bam files and create their index using `samtools`

In [ ]:
%%bash
samtools sort results/HIFI_alignment/PacBio_clover_alignment_1.sam -o results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam
samtools sort results/HIFI_alignment/PacBio_clover_alignment_2.sam -o results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam

In [ ]:
%%bash
samtools index results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam
samtools index results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam

:::{.callout-tip title="Questions - IGV"}

Now you can inspect the alignment files in IGV.

- Remember, you need the references fasta sequence in `../Data/Clover_Data/DNA_Contig1_2.fasta`, `../Data/Clover_Data/DNA_Contig1.fasta`, `../Data/Clover_Data/DNA_Contig2.fasta` imported into IGV. Remember: this is done with the menu `Genomes --> Load Genome from file` and by selecting the relevant fasta file. Then, choose the reference you need from the drop-down menu (see figure below).

&nbsp;

- Load the two bam files obtained with the `map-hifi` settings for `contig1` and `contig2` separately (`results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam` and `results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam`). Look first at those two files:

    - Which differences do you observe?
    - Have a look at their polymorphic regions in IGV. Are they true polymorphisms?

&nbsp;

- Look at the alignment to both contigs: 

    - Why do you see fluctuations in coverage and large regions without any apparent subgenome SNPs? 
    - What are the major differences between the stats for the reads mapped to Contigs1&2 versus contig1 and contig2? 
    - What is your interpretation of the differences?

:::


:::

---

## RNA-seq mapping — short RNA reads

In the `../Data` folder you will find 24 RNA-seq libraries: 12 from the `S10` line and 12 from `Tienshan`. Each library is **paired-end**, which means that each library is split into two files—one for forward reads (R1) and one for reverse reads (R2). For example: `S10_1_1.R1.fastq` and `S10_1_1.R2.fastq`.

We align each library separately, then merge the 12 S10 alignments into one sample and the 12 Tienshan alignments into another. This gives us two final samples for comparison.


Before aligning reads, we need to prepare the reference genome for STAR. This involves two steps:

1. **Build a genome index** using `STAR --runMode genomeGenerate`. This creates a searchable index of the reference genome, allowing STAR to align reads quickly without scanning the whole sequence every time.

2. **Convert gene annotations** from `gff` to `gtf` format using `gffread`. STAR needs annotations in GTF format to count gene-level expression (how many reads map to each gene).

`STAR` is a very complex tool with many options, so it is always useful to [have a reference manual](https://physiology.med.cornell.edu/faculty/skrabanek/lab/angsd/lecture_notes/STARmanual.pdf) (even LLM agents can get STAR wrong because of changing options in new versions and specific combinations of options in specific scenarios).

In [ ]:
%%bash
gffread -T -o reference_data/white_clover_genes.gtf ../Data/Clover_Data/white_clover_genes.gff

In [ ]:
%%bash
STAR --runThreadN 8 \
--runMode genomeGenerate \
--genomeDir results/STAR_output/indexing_contigs_1_2 \
--genomeFastaFiles reference_data/DNA_Contig1_2.fasta \
--sjdbGTFfile reference_data/white_clover_genes.gtf 

We got a warning saying
```
!!!!! WARNING: --genomeSAindexNbases 14 is too large for the genome size=2089554, which may cause seg-fault at the mapping step. Re-run genome generation with recommended --genomeSAindexNbases 9
```
meaning we need shorter strings of bases (9 bases instead of 14) to be indexed, as our reference genome is very short, and too long strings would cause many alignment errors. So we rerun the command with the suggested option (down below).

In [ ]:
%%bash
STAR --runThreadN 8 \
--runMode genomeGenerate \
--genomeDir results/STAR_output/indexing_contigs_1_2 \
--genomeFastaFiles reference_data/DNA_Contig1_2.fasta \
--sjdbGTFfile reference_data/white_clover_genes.gtf \
--genomeSAindexNbases 9

We use again `STAR` to align every single library for `S10`. We extract the library name of each file and run STAR through each pair of files. Note that plant introns are very rarely more than `5000 bp` and that you are mapping to two homoeologous contigs that show high similarity, especially in genic regions. We set the maximum size to 5000 using `--alignIntronMax 5000`.

In [ ]:
%%bash
for i in `ls ../Data/Clover_Data/S10*.R1.fastq`
do

PREFIXNAME=`basename $i .R1.fastq`
echo "###############################################"
echo "##### ALIGNING PAIRED-END READS "$PREFIXNAME
echo "###############################################"
STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ \
--runThreadN 8 \
--runMode alignReads \
--readFilesIn ../Data/Clover_Data/$PREFIXNAME.R1.fastq ../Data/Clover_Data/$PREFIXNAME.R2.fastq \
--outFileNamePrefix results/STAR_output/S10_align_contigs_1_2/$PREFIXNAME \
--outSAMtype BAM SortedByCoordinate \
--outSAMattributes Standard \
--quantMode GeneCounts \
--alignIntronMax 5000

done

Do the same alignment for `Tienshan` libraries

In [ ]:
%%bash
for i in `ls ../Data/Clover_Data/TI*.R1.fastq`
do

PREFIXNAME=`basename $i .R1.fastq`
echo "###############################################"
echo "##### ALIGNING PAIRED-END READS "$PREFIXNAME
echo "###############################################"
STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ \
--runThreadN 8 \
--readFilesIn ../Data/Clover_Data/$PREFIXNAME.R1.fastq ../Data/Clover_Data/$PREFIXNAME.R2.fastq \
--outFileNamePrefix results/STAR_output/TI_align_contigs_1_2/$PREFIXNAME \
--outSAMtype BAM SortedByCoordinate \
--outSAMattributes Standard \
--quantMode GeneCounts \
--alignIntronMax 5000 

done

Run quality control on each aligned library with `MultiQC`. In this way there will be a whole report to compare `S10` files and `Tienshan` files.

In [ ]:
%%bash
multiqc --outdir results/multiqc_output/TI_STAR_align_1_2 \
            results/STAR_output/TI_align_contigs_1_2/

In [ ]:
%%bash
multiqc --outdir results/multiqc_output/S10_STAR_align_1_2 \
            results/STAR_output/S10_align_contigs_1_2/

:::{.callout-tip title="Task"}

Verify the alignments are of good quality by looking at the report statistics

:::

We merge the outputs of each group of aligned libraries. Here is the files for the `Tienshan`.

In [ ]:
!ls -lh  results/STAR_output/TI_align_contigs_1_2/TI_*.sortedByCoord.out.bam 

Apply `samtools merge` to combine the individual per-library BAM files

In [ ]:
%%bash
mkdir -p results/STAR_output/TI_align_contigs_1_2_merge/
samtools merge -f results/STAR_output/TI_align_contigs_1_2_merge/TI.sorted.bam results/STAR_output/TI_align_contigs_1_2/TI_*.sortedByCoord.out.bam 

In [ ]:
%%bash
mkdir -p results/STAR_output/S10_align_contigs_1_2_merge/
samtools merge -f results/STAR_output/S10_align_contigs_1_2_merge/S10.sorted.bam results/STAR_output/S10_align_contigs_1_2/S10_*.sortedByCoord.out.bam 

Index both merging outputs. A file in format `bam.bai` will appear in their respective folders.

In [ ]:
%%bash
samtools index results/STAR_output/S10_align_contigs_1_2_merge/S10.sorted.bam

In [ ]:
%%bash
samtools index results/STAR_output/TI_align_contigs_1_2_merge/TI.sorted.bam

:::{.callout-tip title="Questions - IGV"}

Look at one of the two merged files just created using IGV and the reference for contigs 1 and 2

- How does the visualization differ from the one used for DNA data?
- Have you noticed some junctions are long and overlap existing junctions? Why do you think that happens?
- Load the gene track file (`.gtf`) from the folder `reference_data`. Right-click anywhere in IGV and choose "Sashimi plot". This will show junctions and gene tracks. Zoom into a very crowded section of the plot.
- Click anywhere and set a threshold for minimum junction coverage, for example 20. What happens? Is it cleaner?
- What can this diagnostic be useful for when aligning data like ours?
:::

:::{.callout-note title="Wrapping up"}

In this exercise, you learned to align various types of data after performing quality control for raw data. We looked at some of the options for the aligners and at how to use some of the basic samtools manipulation programs. The outputs from the RNA alignments will be used for the VCF file analysis in the next notebook, and the RNA alignments will be use for the bulk RNA data analysis.

:::